In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------- 1) table as tidy dataframe ----------------
rows = [
    # test_config, training, algo, RMSE, RMSE_score, eff_res, Coastal, HighVar, LowVar, Equatorial
    ("Test 6 nadirs",   "6 nadirs",      "UNet2D",  0.0376,0.7404,201.7754,0.7535,0.9591,0.8311,0.7753),
    ("Test 6 nadirs",   "1-to-7 (univ.)","UNet2D",  0.0376,0.7401,202.6979,0.7534,0.9593,0.8310,0.7749),
    ("Test 2 nadirs",   "6 nadirs",      "UNet2D",  0.0446,0.6919,255.9871,0.6936,0.8879,0.7796,0.7446),
    ("Test 2 nadirs",   "2 nadirs",      "UNet2D",  0.0415,0.7130,237.4825,0.7172,0.9234,0.8030,0.7621),
    ("Test 2 nadirs",   "1-to-7 (univ.)","UNet2D",  0.0415,0.7133,236.9984,0.7185,0.9248,0.8024,0.7602),
    ("Test 2 nadirs",   "—",             "MIOST",   0.0441,0.6949,243.4084,0.6669,0.9083,0.7877,0.7313),
    ("Test 7 nad.+SWOT","6 nadirs",      "UNet2D",  0.0370,0.7441,190.0926,0.7525,0.9636,0.8383,0.7717),
    ("Test 7 nad.+SWOT","1-to-7 (univ.)","UNet2D",  0.0371,0.7433,191.3213,0.7503,0.9630,0.8376,0.7700),
    ("Test 7 nad.+SWOT","7+SWOT",        "UNet3D",  0.0365,0.7479,187.2279,0.7593,0.9666,0.8414,0.7766),
    ("Test 7 nad.+SWOT","7+SWOT",        "4DVarNet-56M",   0.0364,0.7486,188.0354,0.7638,0.9662,0.8412,0.7783),
    ("Test 7 nad.+SWOT","7+SWOT",        "4DVarNet-3D-18M",0.0364,0.7485,186.9046,0.7620,0.9670,0.8416,0.7786),
    ("Test 7 nad.+SWOT","—",             "MIOST",   0.0377,0.7393,197.8449,0.7397,0.9582,0.8355,0.7699),
]
cols = ["test","training","algo","RMSE","RMSE_score","eff_res","Coastal","HighVar","LowVar","Equatorial"]
df = pd.DataFrame(rows, columns=cols)
df["test_n"] = df.test.map({"Test 2 nadirs":2,"Test 6 nadirs":6,"Test 7 nad.+SWOT":7.5})  # 7.5 = 7+SWOT on axis

NAVY,DEEP,TEAL,UNI,SPEC,MIOSTC,VARC = "#21295C","#065A82","#1C7293","#C0392B","#065A82","#9AA9B4","#1E8449"
plt.rcParams.update({"font.size":10,"axes.titleweight":"bold","axes.spines.top":False,"axes.spines.right":False})
fig, ax = plt.subplots(2,2, figsize=(15,10)); fig.suptitle("Universal vs Observation-Specific mapping — UNet / MIOST / 4DVarNet",fontsize=15,fontweight="bold")



In [ ]:
# ---------------- panel A: universal vs specialist (RMSE score, UNet2D) ----------------
a=ax[0,0]
tests=["Test 6 nadirs","Test 2 nadirs","Test 7 nad.+SWOT"]
# (removed unused helper line)
def spec_at(t):  # the matching specialist (trained == tested)
    m={"Test 6 nadirs":"6 nadirs","Test 2 nadirs":"2 nadirs","Test 7 nad.+SWOT":"7+SWOT"}
    r=df[(df.test==t)&(df.training==m[t])&(df.algo=="UNet2D")]
    return r.RMSE_score.values[0] if len(r) else np.nan
def univ_at(t):
    r=df[(df.test==t)&(df.training.str.contains("univ"))&(df.algo=="UNet2D")]
    return r.RMSE_score.values[0] if len(r) else np.nan
x=np.arange(len(tests)); w=0.38
sv=[spec_at(t) for t in tests]; uv=[univ_at(t) for t in tests]
a.bar(x-w/2,sv,w,label="Specialist (trained = tested)",color=SPEC)
a.bar(x+w/2,uv,w,label="Universal (1-to-7)",color=TEAL)
for xi,(s,u) in enumerate(zip(sv,uv)):
    a.text(xi-w/2,s+0.002,f"{s:.4f}",ha="center",fontsize=8)
    a.text(xi+w/2,u+0.002,f"{u:.4f}",ha="center",fontsize=8)
a.set_xticks(x); a.set_xticklabels(["Test 6","Test 2","Test 7+SWOT"]); a.set_ylim(0.66,0.78)
a.set_ylabel("RMSE score (↑)"); a.set_title("A · Universal matches the specialists (UNet2D)"); a.legend(fontsize=8)
a.text(2, 0.665, "(no UNet2D specialist\ntrained @7+SWOT)", ha="center", va="bottom", fontsize=7, color="gray")

# ---------------- panel B: sensitivity to inference config (RMSE, lower=better) ----------------
b=ax[0,1]
order=[2,6,7.5]; xt=["2 nadirs","6 nadirs","7+SWOT"]
def series(mask,label,color,ls="-",mk="o"):
    ys=[]
    for n in order:
        r=df[mask & (df.test_n==n)]
        ys.append(r.RMSE.min() if len(r) else np.nan)   # best available at that config
    b.plot(order,ys,ls,marker=mk,color=color,label=label,lw=2,ms=7)
    return ys
series(df.training.str.contains("univ")&(df.algo=="UNet2D"),"UNet2D universal (1-to-7)",TEAL)
series((df.algo=="UNet2D")&~df.training.str.contains("univ")&(df.training=="6 nadirs"),"UNet2D specialist trained@6",UNI,"--","s")
series(df.algo=="MIOST","MIOST",MIOSTC,":","^")
b.set_xticks(order); b.set_xticklabels(xt); b.invert_xaxis()
b.set_ylabel("RMSE [m] (↓)"); b.set_title("B · Sensitivity to inference constellation"); b.legend(fontsize=8)
b.annotate("fewer obs →",xy=(0.5,0.95),xycoords="axes fraction",ha="center",color="gray")

# ---------------- panel C: the mismatch penalty (Test 2 nadirs) ----------------
c=ax[1,0]
cat=["Specialist\ntrained@6\n(mismatch)","Specialist\ntrained@2\n(matched)","Universal\n1-to-7","MIOST"]
val=[df[(df.test=="Test 2 nadirs")&(df.training=="6 nadirs")].RMSE.values[0],
     df[(df.test=="Test 2 nadirs")&(df.training=="2 nadirs")].RMSE.values[0],
     df[(df.test=="Test 2 nadirs")&(df.training.str.contains("univ"))].RMSE.values[0],
     df[(df.test=="Test 2 nadirs")&(df.algo=="MIOST")].RMSE.values[0]]
colors=[UNI,DEEP,TEAL,MIOSTC]
bars=c.bar(cat,val,color=colors)
for bar,v in zip(bars,val): c.text(bar.get_x()+bar.get_width()/2,v+0.0004,f"{v:.4f}",ha="center",fontsize=9)
c.set_ylabel("RMSE [m] (↓)"); c.set_ylim(0.038,0.046)
c.set_title("C · Specialist mismatch penalty @ Test 2 nadirs")
c.annotate("",xy=(0,val[0]),xytext=(1,val[1]),arrowprops=dict(arrowstyle="<->",color="crimson"))
c.text(0.5,(val[0]+val[1])/2+0.0006,f"{(val[0]-val[1])/val[1]*100:.0f}% worse",ha="center",color="crimson",fontsize=9,fontweight="bold")

# ---------------- panel D: algo head-to-head @ 7+SWOT (radar on regional scores) ----------------
d=ax[1,1]; d.remove(); d=fig.add_subplot(2,2,4,polar=True)
regions=["Coastal","HighVar","LowVar","Equatorial"]
ang=np.linspace(0,2*np.pi,len(regions),endpoint=False); ang=np.concatenate([ang,ang[:1]])
sub=df[(df.test=="Test 7 nad.+SWOT")]
show={"UNet3D":TEAL,"4DVarNet-3D-18M":VARC,"MIOST":MIOSTC}
for algo,col in show.items():
    r=sub[sub.algo==algo]
    if not len(r): continue
    vals=r[regions].values[0]; vals=np.concatenate([vals,vals[:1]])
    d.plot(ang,vals,color=col,lw=2,label=algo); d.fill(ang,vals,color=col,alpha=0.08)
d.set_xticks(ang[:-1]); d.set_xticklabels(regions,fontsize=9); d.set_ylim(0.72,0.98)
d.set_title("D · Regional score @ 7+SWOT (↑)",pad=18); d.legend(loc="lower right",bbox_to_anchor=(1.25,-0.05),fontsize=8)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig("table_viz.png",dpi=150,bbox_inches="tight")
print("saved")